# Research data management with Python and MongoDB (Part 2)

This notebook show an example processing and example analysis procedure for querying raw data from a MongoDB timeseries collection, processing and analysing it for a specific ZBT case. The following steps are performed

1. Preconfiguration: 
    - import relevant libraries
    - define parameters, names, search strings, etc.
    - load environment variables for MongoDB connection
2. Connect to MongoDB and query data
3. Inspect raw data
4. Sample processing on data subset

### 1. Preconfiguration

In [ ]:
# Import relevant libraries
import pandas as pd
import os
import numpy as np
from pymongo import MongoClient, errors
from dotenv import load_dotenv
import chardet
from rich import print
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Define data specific parameters
testbench_name = 'BZ011'
set_id_key = 'Set aktuell'
datetime_key = 'Datum'
database_name = 'rdm_workshop'
collection_name = 'BZ011_Rohdaten'
data_file_path = './data/BZ011_Rohdaten.dat'
datetime_format = "%d.%m.%y %H:%M:%S"
averaging_window = 30

In [ ]:
# Load environment variables (from .env file) for MongoDB connection
load_dotenv()
mongodb_user = os.environ.get("MONGODB_USER")
mongodb_password = os.environ.get("MONGODB_PASS")
mongodb_ip = os.environ.get("MONGODB_IP")
# Check that all user variables were provided in file
if not all([mongodb_user, mongodb_password, mongodb_ip]):
    raise ValueError('Environment variables not correctly set.')

### 2. Connect to MongoDB and query data


In [ ]:
# MongoDB connection
try:
     
    # Select database and collection
    db = client[database_name]
    collection = db[collection_name]
    # Fetch all data from MongoDB collection
    cursor = collection.find({})  # Empty filter `{}` fetches all documents
    # Convert to DataFrame and sort by datetime column
    df_full = pd.DataFrame(list(cursor)).sort_values([datetime_key])
except (errors.ServerSelectionTimeoutError, errors.ConnectionFailure):
    print('MongoDB server not available. Read data from test file instead.')
    with open(data_file_path, "rb") as f:
        result = chardet.detect(f.read(100000))  # Analyze first 100KB
        detected_encoding = result["encoding"]
    df_full = pd.read_csv(data_file_path, delimiter="\t", encoding=detected_encoding,
                    decimal=",")
    df_full['Datum'] = pd.to_datetime(df_full['Datum'], format=datetime_format)

### 3. Inspect raw data


In [ ]:
# Inspect data
df_full.head()

In [ ]:
# Inspect column names
print(df_full.columns)

In [ ]:
# Plot some control data for plausibility checks
fig = px.line(df_full, x='Datum', y=['p_Luft/bar_ein', 'T_Luft_ein', 'Set aktuell', 'U1', 'Spg U / V'])
fig.show()

### 4. Sample processing on data subset
Querying the complete dataset might be suitable for small data sets, however for large data sets it might be required to only query and process subsets of the data due to memory limitations of your computer. Therefore we will now query a subset of the previously loaded full dataset.

For this example we will process all data at a cell voltage of 0.6 V. There is a comment column `"Set Kommentar" = "0,6V"` indicating these operating points. 
We could possibly directly filter the data by a range of cell voltage values in the `"Spg U / V"` column:
`collection.find({"Spg U / V":{"$gte":0.59, "$lt":0.62})`
However there we might also catch unwanted data points at other operating modes.

In [ ]:
# Two different ways of querying for the same data subset
df_subset = pd.DataFrame(list(collection.find({"Set Kommentar": "0,60V"}))).sort_values([datetime_key])
meta_data = df_subset['metadata']
# Metadata should be dropped, since nested lists in dataframes limit functionality of pandas
df_subset = df_subset.drop(columns=['metadata'])
df_subset.info()
df_subset["Set aktuell"]

Now we want to subdivide the data set according to individual continuous operation sections.
Each operation mode is identified by `set_id_key = "Set aktuell"`, however the ids might occur multiple separate times, therefore just grouping by that key is not sufficient and we need further manipulation

In [ ]:
# Calculate the difference between subsequent id values and fill NaN values with 0
set_change = df_subset[set_id_key].diff().fillna(0)
# Change data types to integer
set_change = set_change.astype('int')
# Set all non-zero values to 1
set_change[set_change != 0] = 1
# Build the cumulative sum along the rows to count changing operating modes and add to data frame
df_subset['set_count'] = set_change.cumsum()

In [ ]:
fig = px.scatter(df_subset, x='Datum', y=['set_count', 'Set aktuell', 'U1', 'Spg U / V'])
fig.show()

The next steps include the core of the data processing:
- Data set is filtered for numeric-only data
- The data frame is grouped according to the previously calculated `set_count` column

Further information: https://pandas.pydata.org/docs/user_guide/groupby.html

In [ ]:
# Now group the different sets and apply averaging (last 30 values) individually
grouped_subset = df_subset.select_dtypes(include=np.number).groupby(['set_count'])
grouped_subset.describe()

The first example is using predefined functionality from pandas only:
- Averaging is performed for the last `averaging_window` values of each group with `tail` method
- Since the build-in functions for groups are always aggregation methods (resolving the groups back to dataframes) another groupby-operation is performed
- Then the `mean` method for averaging the reduced data range in each group is applied

Further information: https://pandas.pydata.org/docs/user_guide/groupby.html

In [ ]:
df_averaged = grouped_subset.tail(averaging_window)
df_averaged

In [ ]:
df_averaged = grouped_subset.tail(averaging_window).groupby(['set_count']).mean()
df_averaged

Here a general method is shown if the operations become more complex and involve more steps
- The `apply` method of the GroupBy-objects is used to perform operations on all groups in a single command. Complex operations involving more processing steps for each group can be defined in a separate function, to be passed to the `apply` method. Here a lambda-function is sufficient to define the function (lambda function: anonymous function without explicit naming for use only in a single place). Within the lambda-function:

    - Averaging is performed for the last `averaging_window` values of each group with ´tail´ method, or if the group is smaller then for the last (length_of_group - 2) datapoints to remove transitional data points

    - Then the `mean` method for averaging the selected data range in each group is applied again as before

In [ ]:
# Perform the averaging using the apply method for all groups
df_averaged = grouped_subset.apply(
    lambda x: x.tail(min(averaging_window, len(x) - 2)).mean(), include_groups=False, )
df_averaged.reset_index(inplace=True)
df_averaged

In [ ]:
# More plotting functionality with plotly graph objects instead of plotly express
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add traces
fig.add_trace(go.Scatter(x=df_averaged['set_count'], y=df_averaged['U1'], name="Voltage"),
              secondary_y=False)

fig.add_trace(go.Scatter(x=df_averaged['set_count'], y=df_averaged['Strom I / A'], name="Current"),
              secondary_y=True)

# Add figure title
fig.update_layout(title_text="Double Y Axis Example")

# Set x-axis title
fig.update_xaxes(title_text="Set Count")

# Set y-axes titles
fig.update_yaxes(title_text="Voltage / V", secondary_y=False)
fig.update_yaxes(title_text="Current / A", secondary_y=True)

fig.show()

Now we do a processing where we want to add the processed data into the initial data frame, and fill all averaged rows with its averaged values. Here, we replace the previous `apply` method with the `transform` method to recover the original data frame shape. Further information: https://pandas.pydata.org/docs/user_guide/groupby.html#the-transform-method

In [ ]:
# Perform the averaging using the apply method for all groups
df_averaged = grouped_subset.apply(
    lambda x: x.tail(min(averaging_window, len(x) - 2)).mean(), include_groups=False, )
#df_averaged.reset_index(inplace=True)

# Rename the averaged columns
column_dict = {col: col + '_avg' for col in df_averaged.columns}
df_averaged.rename(columns=column_dict, inplace=True)
new_columns = df_averaged.columns
df_averaged
# df_averaged.set_index(['set_count'], inplace=True)
# df_averaged

In [ ]:
# Get the last rows of each from the full subset to preserve corresponding index and datetimes
df_last_rows_of_groups = df_subset.groupby(['set_count']).tail(1)
#df_last_rows_of_groups.set_index(['set_count'], inplace=True)
df_last_rows_of_groups.reset_index(inplace=True)
df_last_rows_of_groups.set_index(['set_count'], inplace=True)
df_last_rows_of_groups = df_last_rows_of_groups.drop(columns=list(column_dict.keys()))

df_combined_averaged = pd.concat([df_last_rows_of_groups, df_averaged], axis=1)
# df_average = pd.concat([df_last_rows_of_groups, df_averaged], axis=1)
df_combined_averaged.set_index('index', inplace=True)
df_combined_averaged

In [ ]:
# Combine the initial data frame (subset) with the averaged data frame 
df_subset_extended = pd.merge(df_subset, df_combined_averaged, how='outer')
df_subset_extended 

In [ ]:
# Plot raw data subset and corresponding averaged
data = df_subset_extended
x_values = data[datetime_key]
columns = ["set_count", 'U1', 'U1_avg']
y_values = [data[i] for i in columns]

modes = ['lines', 'markers', 'markers']
markers = [
    {'size': 1},
    {'size': 5},
    {'size': 10},
]

fig_2 = go.Figure()
for i in range(len(y_values)):
    fig_2.add_trace(go.Scatter(x=x_values, y=y_values[i], mode=modes[i],
                               marker=markers[i], name=columns[i]))
    # fig_2.add_trace(go.Scatter(x=x_values, y=data['U1_averaged'],
    #                            mode='markers'))
fig_2.update_layout(
    font_family="Arial",
    font_color="black",
    font_size=14,

)
fig_2.update_xaxes(title="DateTime")
fig_2.update_yaxes(title="Value")
fig_2.show()